In [22]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [23]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber_full/"

In [24]:
from mhcgnomes import parse

# read total data

source_total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(source_total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
    
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-Dd', 'H-2-Ld', 'H-2-IEd', 'H-2-Kd', 'H-2-IAd']

# 1. Read all the predictions

In [25]:
import os


In [26]:
per_method_columns = {
    'NetMHC': ['netMHC-3.4_score', 'netMHC-3.4_affinity', 'netMHC-4.0_score',  'netMHC-4.0_affinity','netMHC-4.0_percentile_rank', \
               'netMHCpan-4.1_score_ba', 'netMHCpan-4.1_score_el', 'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 'netMHCpan-4.1_percentile_rank_el',
                'netMHCIIpan-4.0_score_ba', 'netMHCIIpan-4.0_score_el', 'netMHCIIpan-4.0_affinity', 'netMHCIIpan-4.0_percentile_rank_ba', 'netMHCIIpan-4.0_percentile_rank_el',],
    'netMHC-3.4': [ 'score', 'affinity'],
    'netMHC-4.0':[ 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':[ 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0' :['score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],

    'mhcflurry':['affinity', 'percentile_rank_ba', 'processing_score', 'score_el','percentile_rank_el' ],
    'mhcnuggets':[ 'score', 'affinity', 'percentile_rank'],
    'acme':['score'],
    'puffin':['score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood', 'percentile_rank'],
    'mhcseqnet':['score'],
    'mhcattnnet':[ 'score'],
    'ghmm': ['HMMB', 'percentile_rank']
}



In [27]:
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber_full/"
methods =  next(os.walk(PREDICTION_PATH))[1] # read all the folders in a prediction path
methods.remove('NetMHC')
methods.insert(0, 'NetMHC')

#['NetMHC', 'puffin', 'mhcnuggets', 'acme', 'mhcseqnet', 'mhcattnnet']

per_allele_per_method_predictions = dict()

def less_is_good(column):
    if column.find('affinity') >= 0 :
        return True
    if column.find('percentile_rank') >=0:
        return True
    if column in ['epistemic_var', 'aleatoric_var', 'HMMB']:
        return True
    return False

for allele_name in ALLELES:
    print(f"{allele_name}:", end=" ")
    per_allele_per_method_predictions[allele_name] = dict()
    for method in methods:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        prediction_df = pd.read_csv(f"{PREDICTION_PATH}{method}/{method}_predictions_allele_{target_allele_name}.csv", sep=';')

        #add diff 

        print(f"{method}", end=" ")
        for score_column in per_method_columns[method]:
            result_score_column = f"{method}_{score_column}" if not (method == 'NetMHC') else score_column
            wt_scores = prediction_df[result_score_column + '_wt'].values
            mt_scores = prediction_df[result_score_column + '_mt'].values
            wt_peptide =  prediction_df['peptide_wt'].values
            # for these columns mt shoud be less than wt. In this case wt/mt will be > 1 and this is good. 
            if less_is_good(score_column):
                advantage_ratio = np.array([wt_sc/(mt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
                advantage_delta = np.array([wt_sc - mt_sc if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])

        # for these columns mt shoud be greater than wt. In this case mt/wt will be > 1 and this is good.
            else:
                advantage_ratio = np.array([mt_sc/(wt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
                advantage_delta = np.array([mt_sc - wt_sc if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            prediction_df[f"{result_score_column}_advantage_ratio"] = advantage_ratio
            prediction_df[f"{result_score_column}_advantage_delta"] = advantage_delta



        per_allele_per_method_predictions[allele_name][method] = prediction_df
    print()


H-2-Dd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Ld: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-IEd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Kd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-IAd: NetMHC ghmm mhcflurry mhcnuggets puffin 


In [28]:
per_allele_per_method_predictions[ALLELES[1]]['NetMHC']

,source_sequence_name,allele,peptide_mt,length,netMHC-3.4_score_mt,netMHC-3.4_affinity_mt,netMHC-3.4_percentile_rank_mt,netMHC-4.0_score_mt,netMHC-4.0_affinity_mt,netMHC-4.0_percentile_rank_mt,...,netMHCIIpan-4.0_score_ba_advantage_ratio,netMHCIIpan-4.0_score_ba_advantage_delta,netMHCIIpan-4.0_score_el_advantage_ratio,netMHCIIpan-4.0_score_el_advantage_delta,netMHCIIpan-4.0_affinity_advantage_ratio,netMHCIIpan-4.0_affinity_advantage_delta,netMHCIIpan-4.0_percentile_rank_ba_advantage_ratio,netMHCIIpan-4.0_percentile_rank_ba_advantage_delta,netMHCIIpan-4.0_percentile_rank_el_advantage_ratio,netMHCIIpan-4.0_percentile_rank_el_advantage_delta
0,seq1,H-2-Ld,ALSGVSVYF,9,0.037,33498.0,NaN,0.041,32036.15,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,seq2,H-2-Ld,QPAAAPAAAL,10,0.338,1289.0,NaN,0.149,9991.85,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,seq3,H-2-Ld,LSGVSVYF,8,0.070,23375.0,NaN,0.053,28269.03,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,seq4,H-2-Ld,QVLGGVNL,8,0.047,30194.0,NaN,0.084,20206.55,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,seq5,H-2-Ld,AAAPAAAL,8,0.038,33038.0,NaN,0.014,42995.14,37.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,seq17637,H-2-Ld,REPAPEWPSIN,11,0.006,46727.0,NaN,0.110,15260.97,3.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8734,seq17638,H-2-Ld,DKGKIEDTGGS,11,0.005,47178.0,NaN,0.004,47994.24,95.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8735,seq17639,H-2-Ld,NDFSVLPLPN,10,0.006,46677.0,NaN,0.004,47756.99,95.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8736,seq17640,H-2-Ld,EPAPEWPSINL,11,0.368,928.0,NaN,0.291,2139.38,0.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. Merge predictions

In [29]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    list_for_allele = list(per_allele_per_method_predictions[allele_name].values())

    result_df = list_for_allele[0].copy()
    for df_to_merge in list_for_allele[1:]:
        result_df = result_df.merge(df_to_merge, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'], how='left')
    result_df = result_df[['source_sequence_name','allele', 'length'] + [column for column in result_df.columns if column.endswith('mt')] 
                          + [column for column in result_df.columns if column.endswith('_advantage_ratio')]
                          + [column for column in result_df.columns if column.endswith('_advantage_delta')]
                          + [column for column in result_df.columns if column.endswith('wt')]]
    per_allele_predictions[allele_name] = result_df


In [30]:
all_methods = list(set([item.split('_')[0] for item in result_df.columns if  item.split('_')[0] not in ['allele', 'peptide', 'length', 'source']]))
all_methods

['mhcnuggets',
 'netMHC-4.0',
 'ghmm',
 'netMHC-3.4',
 'netMHCpan-4.1',
 'puffin',
 'netMHCIIpan-4.0',
 'mhcflurry']

In [31]:
unsupported_alleles = {method : set() for method in all_methods}

def add_unsupported_allele(method, alleles):
    methods_set = set(all_methods)
    if (method in methods_set):
        unsupported_alleles[method].update(alleles)

for allele_name in ALLELES:
    if defineClass(allele_name) == 'II':
        add_unsupported_allele('netMHCpan-4.1', [allele_name])
        add_unsupported_allele('netMHC-3.4', [allele_name])
        add_unsupported_allele('netMHC-4.0', [allele_name])
        add_unsupported_allele('acme', [allele_name])
        add_unsupported_allele('mhcflurry', [allele_name])
    else:
        add_unsupported_allele('netMHCIIpan-4.0', [allele_name])
        
    if not defineOrganism(allele_name) == 'Human':
        add_unsupported_allele('mhcseqnet', [allele_name])
        add_unsupported_allele('mhcattnnet', [allele_name])
        add_unsupported_allele('acme', [allele_name])

#special cases
add_unsupported_allele('acme',["HLA-C*04:01", "HLA-C*03:04"])
add_unsupported_allele('puffin',["HLA-C*04:01"])


def drop_unsupported_columns(per_allele_df):
    per_allele_all_methods = {allele_name : set() for allele_name in per_allele_df.keys()}

    for allele_name, df in per_allele_df.items():
        for method in all_methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)] + ['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt']
                df = df.drop(columns=columns_to_drop, errors='ignore')
            else:
                per_allele_all_methods[allele_name].add(method)
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df, per_allele_all_methods



# Add source info

In [32]:

source_total_columns = ['VAR_ID', 'mutant_id', 'transcript_id', 'peptide_offset',
                        'A20_RGT', 'A20_RDP', 'A20_RAD_REF',
                        'A20_RAD_ALT', 'A20_RAF_ALT', 'A20_RPKM_mut',  'source_sequence_name', 
                        'peptide_wt', 'peptide_mt' , 'allele',  'length'
                       ]
source_total_df = source_total_df[source_total_columns]
source_total_df

,VAR_ID,mutant_id,transcript_id,peptide_offset,A20_RGT,A20_RDP,A20_RAD_REF,A20_RAD_ALT,A20_RAF_ALT,A20_RPKM_mut,source_sequence_name,peptide_wt,peptide_mt,allele,length
0,4_147755805_T/A,Zfp984-K196N,ENSMUST00000105734,188,T/A,249,95,154,0.618474,5.234282,seq0,KYLTQREKL,KYLTQRENL,H-2-Kd,9
1,18_53706875_A/C,Cep120-L771R,ENSMUST00000049811,765,A/C,259,110,149,0.575290,3.492483,seq6,VYQVELERL,VYQVERERL,H-2-Kd,9
2,11_69804901_G/C,Tmem102-P82A,ENSMUST00000051025,79,G/C,90,54,36,0.400000,1.645296,seq11,RFPPQAELL,RFAPQAELL,H-2-Kd,9
3,3_104800372_T/C,Mov10-D635G,ENSMUST00000166979,627,T/C,181,82,99,0.546961,1.347878,seq14,APSNSGADL,APSNSGAGL,H-2-Kd,9
4,11_60031753_G/A,Pemt-A55V,ENSMUST00000102693,50,G/A,21,15,6,0.285714,0.061122,seq15,SFVAAVITI,SFVAVVITI,H-2-Kd,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130391,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,233,A/G,939,0,939,1.000000,10.112060,seq92523,MCNSSCMGGMNR,MCNGSCMGGMNR,H-2-IAd,12
130392,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,232,A/G,939,0,939,1.000000,10.112060,seq92527,YMCNSSCMGGMNR,YMCNGSCMGGMNR,H-2-IAd,13
130393,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,231,A/G,939,0,939,1.000000,10.112060,seq92528,KYMCNSSCMGGMN,KYMCNGSCMGGMN,H-2-IAd,13
130394,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,231,A/G,939,0,939,1.000000,10.112060,seq92534,KYMCNSSCMGGM,KYMCNGSCMGGM,H-2-IAd,12


In [33]:
per_method_main_scores = {
    'netMHC-3.4': 'affinity',
    'netMHC-4.0': 'affinity',
    'netMHCpan-4.1': 'score_el',
    'netMHCIIpan-4.0' : 'score_el',
    'mhcflurry': 'score_el',
    'mhcnuggets': 'affinity',
    'acme':'score',
    'puffin':'score',
    'mhcseqnet':'score',
    'mhcattnnet': 'score',
    'ghmm': 'HMMB',
}

per_method_main_percentiles = {
    'netMHC-4.0':'percentile_rank' ,
    'netMHCpan-4.1': 'percentile_rank_el',
    'netMHCIIpan-4.0' : 'percentile_rank_el',
    'mhcflurry':'percentile_rank_el' ,
    'mhcnuggets': 'percentile_rank',
    #'acme':'score',
    'puffin': 'percentile_rank',
    #'mhcseqnet': 'score',
    #'mhcattnnet': 'score',
    'ghmm': 'percentile_rank'
}

def reorder_columns(df, source_total_columns, source_id_columns):
    general_info = ['mutant_id', 'allele', 'length', 'median_percentile','mean_percentile',
                    'median_percentile_delta', 'mean_percentile_delta', 'median_score_ratio', 'A20_RPKM_mut']



    source_columns = [source_column for source_column in source_total_columns
                          if source_column not in general_info
                          and source_column not in source_id_columns
                          and source_column not in ['peptide_wt', 'peptide_mt']]


    percentile_columns_mt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('mt')]
    percentile_columns_wt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('wt')]
    percentile_columns = percentile_columns_mt + percentile_columns_wt
    not_percentile_columns = [column for column in df.columns 
                              if column not in percentile_columns 
                              and  column not in general_info 
                              and column not in ['peptide_wt', 'peptide_mt'] 
                              and column not in source_columns]

    columns_ordered = list()
    columns_ordered.extend(general_info)
    columns_ordered.append('peptide_mt')
    columns_ordered.extend(percentile_columns_mt)

    columns_ordered.append('peptide_wt')
    columns_ordered.extend(percentile_columns_wt)
    columns_ordered.extend(source_columns)
    columns_ordered.extend(not_percentile_columns)
    columns_ordered.append(RESULT_COLUMNS['id'])

            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return df[columns_ordered]

def add_avg_percentile_for_selected(per_allele_predictions, selected_methods, per_allele_methods):
    per_allele_result = dict()
    for allele_name, df in per_allele_predictions.items():
        selected_percentiles = [method + "_" + per_method_main_percentiles[method] for method in selected_methods if method in per_allele_methods[allele_name]]
        selected_percentiles_columns = [percentile + "_mt" for percentile in selected_percentiles]
        selected_percentiles_advantage_delta = [percentile + "_advantage_delta" for percentile in selected_percentiles]

        selected_main_scores = [method + "_" + per_method_main_scores[method]  for method in selected_methods if method in per_allele_methods[allele_name]]
        selected_main_scores_advantage_ratio = [score  + "_advantage_ratio" for score in selected_main_scores]

        df['median_percentile'] = df[selected_percentiles_columns].median(axis=1)
        df['mean_percentile'] = df[selected_percentiles_columns].mean(axis=1)
        df['median_percentile_delta'] = df[selected_percentiles_advantage_delta].median(axis=1)
        df['mean_percentile_delta'] = df[selected_percentiles_advantage_delta].mean(axis=1)

        df['median_score_ratio'] = df[selected_main_scores_advantage_ratio].median(axis=1)

        per_allele_result[allele_name] = df
    return per_allele_result




per_allele_predictions, per_allele_methods = drop_unsupported_columns(per_allele_predictions)
per_allele_predictions = add_avg_percentile_for_selected(per_allele_predictions,
                                                         selected_methods = ['netMHCIIpan-4.0', 'netMHCpan-4.1', 'mhcflurry','mhcnuggets','puffin'],
                                                         per_allele_methods=per_allele_methods)




class_I_dfs = list()
class_II_dfs = list()
for allele_name in ALLELES:
    df_allele_source = source_total_df.loc[source_total_df['allele'] == allele_name].copy()
    df_predictions = per_allele_predictions[allele_name]
    df_result = df_predictions.merge(df_allele_source, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'])
    df_result = reorder_columns(df_result, source_total_columns, source_id_columns=['mutand_id'])
    per_allele_predictions[allele_name] = df_result
    if defineClass(allele_name) == 'I':
        class_I_dfs.append(df_result)
    elif defineClass(allele_name) == 'II':
        class_II_dfs.append(df_result)

class_I_total_df = pd.concat(class_I_dfs)
class_II_total_df = pd.concat(class_II_dfs)




In [34]:
per_allele_predictions[ALLELES[0]].columns

Index(['mutant_id', 'allele', 'length', 'median_percentile', 'mean_percentile',
       'median_percentile_delta', 'mean_percentile_delta',
       'median_score_ratio', 'A20_RPKM_mut', 'peptide_mt',
       ...
       'mhcflurry_affinity_wt', 'mhcflurry_processing_score_wt',
       'mhcflurry_score_el_wt', 'mhcnuggets_score_wt',
       'mhcnuggets_affinity_wt', 'puffin_score_wt', 'puffin_epistemic_var_wt',
       'puffin_aleatoric_var_wt', 'puffin_binding_likelihood_wt',
       'source_sequence_name'],
      dtype='object', length=121)

In [35]:
per_allele_methods

{'H-2-Dd': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'},
 'H-2-Ld': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'},
 'H-2-IEd': {'ghmm', 'mhcnuggets', 'netMHCIIpan-4.0', 'puffin'},
 'H-2-Kd': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'},
 'H-2-IAd': {'ghmm', 'mhcnuggets', 'netMHCIIpan-4.0', 'puffin'}}

In [36]:
sum([len(df) for df in per_allele_mt_df.values()])

130396

In [37]:
len(class_I_total_df) + len(class_II_total_df)

130396

In [38]:
len(source_total_df)

130396

## Save total pred to file

In [39]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)

In [40]:
class_I_total_df.to_csv("{}total_predictions_class_I.csv".format(PREDICTION_PATH),sep=";", index=False)
class_II_total_df.to_csv("{}total_predictions_class_II.csv".format(PREDICTION_PATH),sep=";", index=False)

In [41]:
class_I_total_df

,mutant_id,allele,length,median_percentile,mean_percentile,median_percentile_delta,mean_percentile_delta,median_score_ratio,A20_RPKM_mut,peptide_mt,...,mhcflurry_affinity_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,source_sequence_name
0,Serpinf1-I204S,H-2-Dd,8,17.924465,23.038045,-7.286791,-13.638770,0.738053,0.048448,SALSSLLL,...,13139.408338,0.168184,0.013483,0.086906,19525.50,0.119096,0.000718,0.018987,3.899331e-02,seq124
1,Fndc7-S519F,H-2-Dd,9,58.103337,56.234918,2.318000,1.352625,0.975702,0.026786,FVTAVAETQ,...,24284.059134,0.024910,0.004513,0.029079,36502.88,0.023569,0.000114,0.001563,2.630989e-07,seq194
2,Fndc7-S519F,H-2-Dd,8,68.538875,68.570526,2.246750,0.650375,1.032751,0.026786,FVTAVAET,...,29054.427592,0.020173,0.003731,0.028266,36825.66,0.020565,0.000108,0.001339,2.134040e-08,seq329
3,Kntc1-D910A,H-2-Dd,8,5.106875,14.344168,0.455500,-1.692156,1.073558,7.567225,GGACLLLL,...,16777.426978,0.022058,0.006387,0.186651,6635.94,0.207015,0.006835,0.037774,1.218492e-01,seq401
4,Ruvbl2-R253C,H-2-Dd,8,57.334875,56.378418,-1.955250,-6.660781,1.002838,13.436731,DVINSCTQ,...,28488.109146,0.173762,0.006531,0.037985,33149.65,0.016420,0.000055,0.000985,2.431140e-09,seq431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,Sephs1-A176P,H-2-Kd,8,95.656625,88.915526,-2.500000,-2.360250,0.936406,4.831458,PTTVCQPN,...,31587.679473,0.001968,0.003227,0.036197,33797.39,0.071430,0.000381,0.009629,8.161671e-03,seq14195
8734,Gm1141-L342R,H-2-Kd,8,88.201875,81.904901,-7.701000,-9.461500,0.795284,0.039319,PRLHHSSR,...,29461.855873,0.005839,0.003500,0.080422,20944.43,0.058461,0.000361,0.007093,3.540867e-03,seq14197
8735,Gm1141-L342R,H-2-Kd,11,71.843375,73.163776,-5.535500,-5.035500,0.917218,0.039319,PPRLHHSSRSC,...,28689.863950,0.001152,0.003532,0.057496,26841.04,0.095455,0.000500,0.015510,2.651579e-02,seq14200
8736,Ssbp3-G353A,H-2-Kd,8,80.335375,78.050276,3.508750,5.815000,1.045723,1.247553,PNNISAIS,...,31438.486048,0.001649,0.003239,0.034705,34347.13,0.079884,0.000562,0.011324,1.167089e-02,seq14201


In [42]:
per_allele_predictions[allele_name]

,mutant_id,allele,length,median_percentile,mean_percentile,median_percentile_delta,mean_percentile_delta,median_score_ratio,A20_RPKM_mut,peptide_mt,...,netMHCIIpan-4.0_score_el_wt,netMHCIIpan-4.0_affinity_wt,ghmm_HMMB_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,source_sequence_name
0,Apoe-V184L,H-2-IAd,13,17.15725,14.661833,0.02000,-0.020000,1.015333,0.385506,EGAERGLSAIRER,...,0.827304,2902.48,24.695894,0.252915,3239.89,0.198457,0.007626,0.025252,0.076421,seq11
1,Ybx1-P11A,H-2-IAd,14,0.29125,0.718167,0.79200,0.589167,1.080103,91.274678,QPAAAPAAALSAAD,...,0.769551,762.73,25.498467,0.528704,163.91,0.431860,0.008535,0.029878,0.514388,seq14
2,Apoe-V184L,H-2-IAd,14,6.24925,8.508333,0.00000,-0.094333,0.999502,0.385506,EGAERGLSAIRERL,...,0.841370,1000.82,24.878348,0.384247,782.36,0.236598,0.006031,0.027805,0.128482,seq16
3,Ybx1-P11A,H-2-IAd,13,1.83925,1.555333,1.22350,1.619667,1.092252,91.274678,QPAAAPAAALSAA,...,0.643526,1810.04,25.810546,0.373493,878.90,0.418411,0.009842,0.030926,0.483638,seq17
4,Apoe-V184L,H-2-IAd,14,13.71450,10.125250,0.01000,0.249750,1.010260,0.385506,REGAERGLSAIRER,...,0.832941,1430.57,24.910407,0.275817,2528.79,0.247490,0.007101,0.028147,0.144169,seq18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52086,Trp53-S237G,H-2-IAd,12,89.94225,84.574000,-4.52250,-8.536500,0.731899,10.112060,MCNGSCMGGMNR,...,0.000076,21070.17,41.890343,0.108507,15456.05,0.041586,0.000925,0.007130,0.000003,seq92523
52087,Trp53-S237G,H-2-IAd,13,69.60625,67.909833,-8.75950,-5.976333,0.699589,10.112060,YMCNGSCMGGMNR,...,0.000647,14069.36,41.789657,0.167082,8200.85,0.109384,0.006971,0.016590,0.007039,seq92527
52088,Trp53-S237G,H-2-IAd,13,65.06575,69.573333,-0.38075,-2.800500,0.871977,10.112060,KYMCNGSCMGGMN,...,0.001479,12726.12,41.473260,0.069825,23488.93,0.158280,0.012218,0.021797,0.035084,seq92528
52089,Trp53-S237G,H-2-IAd,12,66.99325,70.119333,-3.39150,-3.084000,0.858993,10.112060,KYMCNGSCMGGM,...,0.000686,14961.13,41.993698,0.116052,14244.40,0.107662,0.007560,0.016833,0.007128,seq92534
